# 다시 보기 — `pd.concat()` · 선형 회귀 · Lasso

수업 중 다시 짚은 `pd.concat()` 과 `선형회귀실습`(2개) 필기를 **실행 가능하도록 하나로 정리 · 개선한 노트**입니다.
(교재 `머신러닝` v5 p.26~41 — 처음 배운 내용은 [Day 020](../../2026-10-06/) concat · [Day 021](../../2026-10-07/) 선형 회귀)

- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.
- 그림 · **애니메이션(GIF)** 은 [`images/`](./images/) 폴더에 미리 만들어 두었습니다.
- 오늘의 새 내용인 부스팅 · 스태킹은 [`boosting_xgboost.ipynb`](./boosting_xgboost.ipynb) · [`stacking.ipynb`](./stacking.ipynb) 에 정리했습니다.

> 실습 데이터 `house_price.csv` (보스턴 주택 가격, 506행) 는 이 노트북과 **같은 폴더**에 있어야 합니다.

In [ ]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. `pd.concat()` — 데이터프레임 병합 (교재 p.26~28)

| 옵션 | 동작 |
|------|------|
| `axis=0` (기본값) | **공통 컬럼**을 기준으로 **아래(y축)** 로 이어 붙임 |
| `axis=1` | **공통 행 인덱스**를 기준으로 **옆(x축)** 으로 이어 붙임 — 인덱스가 다르면 `NaN` |
| `ignore_index=True` | 원래 인덱스를 버리고 0 부터 새로 매김 |
| `join='outer'` (기본) / `'inner'` | 컬럼이 다를 때 **합집합** / **공통 컬럼만** |

In [ ]:
import pandas as pd

# 컬럼(A~D)은 같고, 행 인덱스만 다른 DataFrame 3개 (교재 p.26)
df1 = pd.DataFrame({'A': ['A0', 'A1', 'A2', 'A3'], 'B': ['B0', 'B1', 'B2', 'B3'],
                    'C': ['C0', 'C1', 'C2', 'C3'], 'D': ['D0', 'D1', 'D2', 'D3']}, index=[0, 1, 2, 3])
df2 = pd.DataFrame({'A': ['A4', 'A5', 'A6', 'A7'], 'B': ['B4', 'B5', 'B6', 'B7'],
                    'C': ['C4', 'C5', 'C6', 'C7'], 'D': ['D4', 'D5', 'D6', 'D7']}, index=[4, 5, 6, 7])
df3 = pd.DataFrame({'A': ['A8', 'A9', 'A10', 'A11'], 'B': ['B8', 'B9', 'B10', 'B11'],
                    'C': ['C8', 'C9', 'C10', 'C11'], 'D': ['D8', 'D9', 'D10', 'D11']}, index=[8, 9, 10, 11])

# axis=0 (기본값) : 아래로 → 4 + 4 + 4 = 12행
df_concat = pd.concat([df1, df2, df3], ignore_index=True)
print("axis=0 :", df_concat.shape)
df_concat

In [ ]:
# axis=1 : 옆으로 → 행 인덱스가 겹치지 않으면 빈칸이 NaN
print(pd.concat([df1, df2], axis=1))

print("-" * 50)

# 인덱스가 같은 두 표를 옆으로 붙이는 실전 예 : 같은 사람의 다른 특성 합치기
print(pd.concat([df1[['A', 'B']], df1[['C', 'D']]], axis=1))

In [ ]:
# 컬럼이 다를 때 : df4 는 B, D, F 컬럼 (A, C 없음 / F 는 새 컬럼)
df4 = pd.DataFrame({'B': ['B2', 'B3', 'B6', 'B7'], 'D': ['D2', 'D3', 'D6', 'D7'],
                    'F': ['F2', 'F3', 'F6', 'F7']}, index=[2, 3, 6, 7])

print("join='outer' (기본, 합집합) :", list(pd.concat([df1, df4], ignore_index=True).columns))
print("join='inner' (교집합)       :", list(pd.concat([df1, df4], join='inner', ignore_index=True).columns))

> - `ignore_index=False`(기본값)면 원래 인덱스가 그대로 남아서 **같은 번호가 두 번** 나올 수 있다 → 합친 뒤 `loc[번호]` 가 여러 행을 돌려준다
> - `axis=1` 은 **행 인덱스가 같은 대상**(같은 사람 · 같은 날짜)일 때만 의미가 있다

## 2. 선형 회귀 다시 보기 (교재 p.29~38)

**회귀** — 처음 보는 데이터의 값을 **연속적인 숫자**로 예측. **선형 회귀** — 데이터를 가장 잘 표현하는 직선 $y = Wx + b$ 를 찾는 것.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression

# 공부 시간(x) → 시험 성적(y) (교재 p.30)
X_hour = np.array([[9], [8], [4], [2]])    # scikit-learn 의 X 는 반드시 2차원 (행 : 샘플, 열 : 특성)
y_score = np.array([90, 80, 40, 20])

model = LinearRegression().fit(X_hour, y_score)
print(f"기울기 W = {model.coef_[0]:.2f}, 절편 b = {round(model.intercept_, 2) + 0.0:.2f}")   # + 0.0 : -0.00 표시 방지
print(f"7시간 공부하면 → {model.predict([[7]])[0]:.1f}점")

| 지표 | 공식 | 특징 |
|------|------|------|
| **MAE** | $\sum \lvert y - \hat{y} \rvert / n$ | 오차의 절댓값 평균 — 이상치에 덜 민감 |
| **MSE** | $\sum (y - \hat{y})^2 / n$ | 큰 오차에 큰 벌점 — 단위가 제곱 |
| **RMSE** | $\sqrt{MSE}$ | **정답과 같은 단위**로 읽는다 |

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

y_true = np.array([3.0, 5.0, 2.5, 7.0])
y_pred = np.array([2.5, 5.0, 4.0, 8.0])

# 공식으로 직접 계산 vs scikit-learn 함수
print(f"MAE  : 직접 {np.mean(np.abs(y_true - y_pred)):.4f} | sklearn {mean_absolute_error(y_true, y_pred):.4f}")
print(f"MSE  : 직접 {np.mean((y_true - y_pred) ** 2):.4f} | sklearn {mean_squared_error(y_true, y_pred):.4f}")
print(f"RMSE : 직접 {np.sqrt(np.mean((y_true - y_pred) ** 2)):.4f} | sklearn {root_mean_squared_error(y_true, y_pred):.4f}")

### 보스턴 집값 — 분할 · 표준화 · 학습 · 평가 (교재 p.39~41)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error as rmse

df = pd.read_csv('./house_price.csv')
print("모양 :", df.shape, "/ 누락 데이터 :", df.isnull().sum().sum(), "개")

X_data = df.drop(columns=['medv'])     # 특성 13개
y_data = df['medv']                    # 정답 : 집값 (천 달러)

# 75 : 25 — 회귀는 정답이 연속값이라 stratify 를 쓰지 않는다
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.25, random_state=0)

# 표준화 : 학습용으로 기준(평균 · 표준편차)을 잡고, 평가용은 그 기준으로 변환만 (데이터 누수 방지)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lireg = LinearRegression().fit(X_train_scaled, y_train)
pred_lr = lireg.predict(X_test_scaled)

print(f"LinearRegression → MAE {mean_absolute_error(y_test, pred_lr):.2f} · RMSE {rmse(y_test, pred_lr):.2f}")

> RMSE **5.46** → 평균적으로 약 5,460달러 틀렸다. ([Day 021](../../2026-10-07/) 과 같은 분할 · 같은 값)

## 3. Lasso (L1 규제) — `alpha` 가 커지면 (교재 p.42~45 복습)

기울기에 **벌점**을 줘서 모델을 단순하게 → 약한 특성의 기울기는 **0** 이 된다(특성 선택). `alpha` 가 클수록 규제가 세다.

![Lasso alpha 애니메이션](./images/anim_01_lasso_alpha.gif)

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0).fit(X_train_scaled, y_train)

# 기울기 비교 : LinearRegression vs Lasso
coef = pd.DataFrame({'LinearRegression': lireg.coef_, 'Lasso(alpha=1)': lasso.coef_}, index=X_data.columns)
print(coef.round(3))
print("-" * 40)
print("Lasso 가 0 으로 만든 특성 :", list(coef.index[coef['Lasso(alpha=1)'] == 0]))
print(f"Lasso RMSE : {rmse(y_test, lasso.predict(X_test_scaled)):.2f}")

In [ ]:
# alpha 를 바꿔 가며 비교
rows = []
for alpha in [0.01, 0.05, 0.1, 0.3, 1.0, 2.0]:
    m = Lasso(alpha=alpha).fit(X_train_scaled, y_train)
    rows.append({'alpha': alpha, '0 인 기울기 수': int((m.coef_ == 0).sum()),
                 'RMSE': round(rmse(y_test, m.predict(X_test_scaled)), 3)})
pd.DataFrame(rows)

> - `alpha=1.0` 은 13개 중 **8개**를 0 으로 만들고 RMSE **5.96** — 이 데이터에는 규제가 너무 셌다
> - `alpha` 를 0.01 까지 줄이면 0 이 되는 기울기가 없고 RMSE 5.47 로 선형 회귀(5.46)와 거의 같다
> - 이 데이터는 특성(13개)에 비해 샘플(379개)이 충분해서 과대적합이 심하지 않다 → 규제의 이득이 작다

> **📝 고친 부분** (`선형회귀실습`)
> - 경로만 다른 노트북 **2개** + Lasso 버전을 하나로 합침
> - 절대 경로 `C:\Users\…` → `./house_price.csv`, `warnings.filterwarnings('ignore')` 제거
> - 절편이 `-0.00` 으로 찍히던 것 → `+ 0.0` 으로 `0.00`
> - LinearRegression vs Lasso 기울기 비교, `alpha` 비교 추가
> - `concat` 필기의 `join` 옵션 · `ignore_index=False` 주의점 추가

## 4. 정리

| 내용 | 핵심 |
|------|------|
| `pd.concat` | `axis=0` 아래 · `axis=1` 옆 · `ignore_index` · `join` |
| 선형 회귀 | $y = Wx + b$, 학습 = MSE 최소, 평가 = MAE · MSE · **RMSE** |
| 표준화 | 학습용 `fit_transform`, 평가용 `transform` |
| Lasso | `alpha` ↑ → 기울기 0 ↑ (보스턴은 `alpha=1` 이 너무 셈) |